# logsumexp-cross-entropy — worked example 1: Stable cross-entropy with sum reduction

> Worked example from [Delta Drills](https://delta-drills.vercel.app). Atom: `logsumexp-cross-entropy`.

**This is a worked example — read it, run each cell, and follow the reasoning.** It's study material, so there's nothing to submit here. Delta Drills hands you a hands-on version to complete yourself as you get comfortable with the idea.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)
# === manual autograd primitives — shared across all drills in this folder ===
from dataclasses import dataclass, field
from typing import Any, Callable, Optional

grad_tracking_enabled = True

@dataclass
class Recipe:
    func: Optional[Callable] = None
    args: tuple = ()
    kwargs: dict = field(default_factory=dict)
    parents: dict = field(default_factory=dict)

class MiniTensor:
    """Minimal Tensor wrapper for the ARENA-style manual-autograd drills.
    Wraps a raw `torch.Tensor` in `.array`. Carries optional `.recipe`,
    `.requires_grad`, and `.grad` (the accumulated gradient at leaves)."""
    def __init__(self, array, requires_grad: bool = False, recipe=None):
        self.array = array
        self.requires_grad = requires_grad
        self.recipe = recipe
        self.grad = None
    def __repr__(self):
        return f'MiniTensor({self.array!r}, requires_grad={self.requires_grad})'

## Concept

_First time on this topic? Run the **Setup** cell above and skim it: every class and helper mentioned below is defined there. You don't need to have done any other drill first._

Cross-entropy per sample is `logsumexp(logits[i]) - logits[i, target[i]]`. Using `torch.logsumexp` subtracts the per-row max internally, so the formula is stable even for huge logits where naive softmax would overflow. Here we sum (not average) the per-sample losses.

## Worked solution

We build a sum-reduced stable cross-entropy.

1. **First term.** `lse = torch.logsumexp(logits, dim=-1)` gives a `(B,)` vector, the stable `log(sum(exp(row)))`.
2. **Second term.** We pick the target logit per row with advanced indexing `logits[arange(B), target]`, also `(B,)`.
3. **Per-sample loss.** `lse - picked`. This equals `-log(softmax(logits)[target])` but never overflows.
4. **Reduce.** We `.sum()` over the batch to get a 0-D scalar (the sum-reduced loss).

The demo uses logits with one entry at 5000 to show stability, and prints the loss is finite and matches a per-row hand computation.

In [ ]:
import torch as t

t.manual_seed(0)

def cross_entropy_sum(logits, target):
    lse = t.logsumexp(logits, dim=-1)
    B = logits.shape[0]
    picked = logits[t.arange(B), target]
    return (lse - picked).sum()

logits = t.tensor([[0.0, 1.0, 5000.0],
                   [2.0, 3.0, 1.0]])
target = t.tensor([2, 1])
loss = cross_entropy_sum(logits, target)
print('loss finite:', t.isfinite(loss).item(), '| value:', round(loss.item(), 4))
print('scalar shape:', tuple(loss.shape))